# Лекция: Подбор вида зависимости (сравнение регрессий)

**Дисциплина:** Введение в анализ больших данных

Сравниваем три формы связи $y$ и $x$:

1. **Линейная:** $y = a_0 + a_1 x$
2. **Полиномиальная 2-й степени:** $y = a_0 + a_1 x + a_2 x^2$
3. **Логарифмическая:** $y = a_0 + a_1 \log(x)$

Критерии: **MSE**, **AIC**, **Adj. R²**, для вложенных моделей — **ANOVA**, плюс вид остатков.

Демо: «насыщение» урожайности от дозы удобрения (синтетика). Примеры **не из лабораторного задания** — задание выполните на своих данных.


## 0. Импорт


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(21)
print("Библиотеки загружены")


---
## 1. Данные и график

$x$ — доза удобрения, $y$ — урожайность (условные единицы).  
Связь **нелинейная** (насыщение), поэтому линейная модель заведомо слабее.


In [ ]:
dose = np.array([1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 20, 25, 30, 35, 40])
yield_ = 30 + 25 * (1 - np.exp(-dose / 8)) + np.random.normal(0, 1.2, len(dose))

df = pd.DataFrame({"dose": dose, "yield_": yield_})
print(df.round(2).head(10))
print("n =", len(df))


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["dose"], df["yield_"], s=60, edgecolors="k", zorder=3)
plt.xlabel("dose")
plt.ylabel("yield")
plt.title("Урожайность от дозы удобрения")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 2. Три модели

| Модель | Формула в `ols` |
|--------|-----------------|
| Линейная | `"yield_ ~ dose"` |
| Полином 2 | `"yield_ ~ dose + I(dose**2)"` |
| Логарифм | `"yield_ ~ np.log(dose)"` |

Для логарифма нужно $x > 0$.


In [ ]:
m_lin = smf.ols("yield_ ~ dose", data=df).fit()
m_poly = smf.ols("yield_ ~ dose + I(dose ** 2)", data=df).fit()
m_log = smf.ols("yield_ ~ np.log(dose)", data=df).fit()

print("=== Линейная ===")
print(m_lin.summary().tables[1])
print("\n=== Полином 2 ===")
print(m_poly.summary().tables[1])
print("\n=== Логарифмическая ===")
print(m_log.summary().tables[1])


### Кривые на одном графике


In [ ]:
x_grid = np.linspace(df["dose"].min(), df["dose"].max(), 200)
grid = pd.DataFrame({"dose": x_grid})

plt.figure(figsize=(9, 5))
plt.scatter(df["dose"], df["yield_"], s=55, edgecolors="k", label="данные", zorder=3)
plt.plot(x_grid, m_lin.predict(grid), lw=2, label="линейная")
plt.plot(x_grid, m_poly.predict(grid), lw=2, label="полином 2")
plt.plot(x_grid, m_log.predict(grid), lw=2, label="логарифм")
plt.xlabel("dose")
plt.ylabel("yield")
plt.title("Сравнение моделей")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. MSE, AIC, Shapiro остатков

- **MSE** = среднее квадратов остатков  
- **AIC** = `model.aic` (меньше — лучше)  
- Shapiro по остаткам — грубая проверка нормальности


In [ ]:
def diagnose(model, name):
    resid = model.resid
    mse = np.mean(resid ** 2)
    W, p_sw = stats.shapiro(resid)
    print(f"=== {name} ===")
    print(f"  R²      = {model.rsquared:.4f}")
    print(f"  Adj.R²  = {model.rsquared_adj:.4f}")
    print(f"  MSE     = {mse:.4f}")
    print(f"  AIC     = {model.aic:.2f}")
    print(f"  Shapiro p = {p_sw:.4f}")
    print()
    return {
        "model": name,
        "R2": model.rsquared,
        "AdjR2": model.rsquared_adj,
        "MSE": mse,
        "AIC": model.aic,
        "Shapiro_p": p_sw,
    }

cmp = pd.DataFrame([
    diagnose(m_lin, "линейная"),
    diagnose(m_poly, "полином 2"),
    diagnose(m_log, "логарифм"),
])
print(cmp.round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, model, title in zip(
    axes, [m_lin, m_poly, m_log], ["линейная", "полином 2", "логарифм"]
):
    ax.scatter(model.fittedvalues, model.resid, edgecolors="k", alpha=0.75)
    ax.axhline(0, color="red", ls="--")
    ax.set_xlabel("Fitted")
    ax.set_ylabel("Residuals")
    ax.set_title(title)
plt.suptitle("Остатки vs fitted", y=1.02)
plt.tight_layout()
plt.show()


---
## 4. ANOVA для вложенных моделей

Линейная ⊂ полином 2-й степени.  
`anova_lm(m_lin, m_poly)` проверяет, значимо ли добавление $x^2$.


In [ ]:
print("ANOVA: линейная vs полином 2")
print(anova_lm(m_lin, m_poly))
print()
print("Ранжирование по AIC (меньше лучше):")
print(cmp[["model", "AIC", "MSE", "AdjR2"]].sort_values("AIC").to_string(index=False))


---
## 5. Как выбрать модель

1. **AIC** — универсальный критерий (меньше лучше).  
2. **MSE / Adj. R²** — качество подгонки.  
3. **ANOVA** — только для вложенных.  
4. **Остатки** — без тренда и «веера».

Итог по AIC:


In [ ]:
best = cmp.loc[cmp["AIC"].idxmin(), "model"]
print(f"По AIC предпочтительна: {best}")
print(cmp.sort_values("AIC")[["model", "AIC", "MSE", "AdjR2"]].round(4).to_string(index=False))


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Линейная | `smf.ols("y ~ x", data=df).fit()` |
| Полином 2 | `smf.ols("y ~ x + I(x**2)", data=df).fit()` |
| Логарифм | `smf.ols("y ~ np.log(x)", data=df).fit()` |
| MSE | `np.mean(model.resid**2)` |
| AIC | `model.aic` |
| ANOVA (вложенные) | `anova_lm(m1, m2)` |
| Прогноз на сетке | `model.predict(pd.DataFrame({"x": grid}))` |

---
## Что сделать после лекции

1. Повторите три модели на **других** $x$, $y$ (или своём CSV).
2. Откройте лабораторное задание и сравните модели **самостоятельно**.
3. Для логарифма убедитесь, что $x > 0$.

Удачи!
